In [1]:
# ===================================================== #
#                                                       #
# PROYECTO DE DETECCIÓN DE FRACTURAS ÓSEAS EN RAYOS X   #
#                                                       #
# ===================================================== #

# En este codigo haremos:
# 1. Preparar y cargar imágenes desde carpetas organizadas por clases (Fracturado / No Fracturado).
# 2. Entrenar un modelo de red neuronal convolucional para clasificar imágenes.
# 3. Evaluar el desempeño del modelo.
# 4. Visualizar resultados como pérdidas por época, matriz de confusión, etc.
# 5. Guardar los gráficos generados en una carpeta específica.
# 6. Validaacion cruzada para ajustar el modelo.

import os
import torch
import torchvision
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from sklearn.model_selection import KFold

# Ruta al dataset

ruta_dataset = r"C:\Users\j2rod\Documents\MASTER IA\Modulo 4\Ejercicio 4.1\X-ray Imaging Dataset for Detecting Fractured vs. Non-Fractured Bones\Original Dataset"
ruta_guardado = os.path.dirname(ruta_dataset)

# Transformaciones
transformaciones_entrenamiento = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

transformaciones_prueba = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Dataset

dataset_completo = datasets.ImageFolder(ruta_dataset, transform=transformaciones_entrenamiento)
porcentaje_entrenamiento = 0.8
tamano_entrenamiento = int(porcentaje_entrenamiento * len(dataset_completo))
tamano_prueba = len(dataset_completo) - tamano_entrenamiento
conjunto_entrenamiento, conjunto_prueba = random_split(dataset_completo, [tamano_entrenamiento, tamano_prueba])
dataset_completo.transform = transformaciones_entrenamiento
conjunto_prueba.dataset.transform = transformaciones_prueba

etiquetas_entrenamiento = [conjunto_entrenamiento[i][1] for i in range(len(conjunto_entrenamiento))]
clases, conteos = np.unique(etiquetas_entrenamiento, return_counts=True)
pesos_clases = 1. / torch.tensor(conteos, dtype=torch.float)
pesos_muestras = [pesos_clases[etiqueta] for etiqueta in etiquetas_entrenamiento]
sampler_entrenamiento = WeightedRandomSampler(pesos_muestras, len(pesos_muestras))

cargador_entrenamiento = DataLoader(conjunto_entrenamiento, batch_size=32, sampler=sampler_entrenamiento)
cargador_prueba = DataLoader(conjunto_prueba, batch_size=32, shuffle=False)

# Modelo

class ModeloRayosX(nn.Module):
    def __init__(self):
        super(ModeloRayosX, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(-1, 64 * 16 * 16)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

# Entrenamiento

def entrenar_modelo(modelo, cargador, criterio, optimizador, epocas):
    historial_perdida = []
    for epoca in range(epocas):
        modelo.train()
        perdida_total = 0
        for imagenes, etiquetas in cargador:
            imagenes, etiquetas = imagenes.to(dispositivo), etiquetas.to(dispositivo)
            optimizador.zero_grad()
            salida = modelo(imagenes)
            perdida = criterio(salida, etiquetas)
            perdida.backward()
            optimizador.step()
            perdida_total += perdida.item()
        historial_perdida.append(perdida_total)
        print(f" Epoc {epoca+1}/{epocas}, pérdida acumulada: {perdida_total:.4f}")
    return historial_perdida

# Evaluación

def evaluar_modelo(modelo, cargador):
    modelo.eval()
    predicciones_totales = []
    etiquetas_totales = []
    with torch.no_grad():
        for imagenes, etiquetas in cargador:
            imagenes = imagenes.to(dispositivo)
            salida = modelo(imagenes)
            _, pred = torch.max(salida, 1)
            predicciones_totales.extend(pred.cpu().numpy())
            etiquetas_totales.extend(etiquetas.numpy())
    print("\n Reporte de clasificación:")
    print(classification_report(etiquetas_totales, predicciones_totales))
    return etiquetas_totales, predicciones_totales

# Validación cruzada

def validacion_cruzada(dataset, k=5):
    print(f"\n Iniciando validación cruzada con {k} pliegues...")
    kfold = KFold(n_splits=k, shuffle=True, random_state=42)
    resultados = []

    mejores_metricas = {"pliegue": -1, "accuracy": 0, "f1_macro": 0}

    for i, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
        print(f"\n Pliegue {i+1}/{k}")
        train_subset = torch.utils.data.Subset(dataset, train_idx)
        val_subset = torch.utils.data.Subset(dataset, val_idx)

        etiquetas = [train_subset[i][1] for i in range(len(train_subset))]
        clases, conteos = np.unique(etiquetas, return_counts=True)
        pesos_clases = 1. / torch.tensor(conteos, dtype=torch.float)
        pesos_muestras = [pesos_clases[etiqueta] for etiqueta in etiquetas]
        sampler = WeightedRandomSampler(pesos_muestras, len(pesos_muestras))

        train_loader = DataLoader(train_subset, batch_size=32, sampler=sampler)
        val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

        modelo = ModeloRayosX().to(dispositivo)
        pesos_loss = torch.tensor([1.5, 1.0]).to(dispositivo)
        criterio = nn.CrossEntropyLoss(weight=pesos_loss)
        optimizador = optim.Adam(modelo.parameters(), lr=0.001)
        entrenar_modelo(modelo, train_loader, criterio, optimizador, epocas=10)

        etiquetas_val, predicciones_val = evaluar_modelo(modelo, val_loader)
        acc = accuracy_score(etiquetas_val, predicciones_val)
        f1 = f1_score(etiquetas_val, predicciones_val, average='macro')
        print(f"Accuracy: {acc:.4f} | F1 Macro: {f1:.4f}")

        if f1 > mejores_metricas["f1_macro"]:
            mejores_metricas = {"pliegue": i+1, "accuracy": acc, "f1_macro": f1}

        resultados.append((etiquetas_val, predicciones_val))

    print("\n Validación cruzada finalizada.")
    print(f"\n Mejor pliegue: {mejores_metricas['pliegue']} | Accuracy: {mejores_metricas['accuracy']:.4f} | F1 Macro: {mejores_metricas['f1_macro']:.4f}")

# Gráficos
    
def mostrar_y_guardar_grafico(figura, nombre):
    guardar = input("¿Deseas guardar este gráfico como imagen? (s/n): ")
    if guardar.lower() == 's':
        ruta = os.path.join(ruta_guardado, f"{nombre} - Fractured vs. Non-Fractured Bones.png")
        figura.savefig(ruta, dpi=300)
        print(f"Gráfico guardado en: {ruta}")
    plt.show()

# Menú

def menu():
    print("""
MENÚ DEL PROYECTO DE ANÁLISIS DE RAYOS X
1. Entrenar el modelo
2. Evaluar el modelo
3. Ver gráfico de pérdida
4. Ver matriz de confusión
5. Mostrar imágenes clasificadas
6. Validación cruzada
7. Salir
""")

# Ejecución

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo = ModeloRayosX().to(dispositivo)
pesos_loss = torch.tensor([1.5, 1.0]).to(dispositivo)
criterio = nn.CrossEntropyLoss(weight=pesos_loss)
optimizador = optim.Adam(modelo.parameters(), lr=0.001)
perdidas = []
etiquetas_test, predicciones_test = [], []

while True:
    menu()
    opcion = input("Selecciona una opción (1-7): ")

    if opcion == '1':
        print("\n Iniciando entrenamiento del modelo...")
        perdidas = entrenar_modelo(modelo, cargador_entrenamiento, criterio, optimizador, epocas=20)
    elif opcion == '2':
        print("\n Evaluando el modelo con datos nuevos...")
        etiquetas_test, predicciones_test = evaluar_modelo(modelo, cargador_prueba)
    elif opcion == '3':
        if not perdidas:
            print("Debes entrenar el modelo primero (opción 1).")
        else:
            fig = plt.figure()
            plt.plot(range(1, len(perdidas)+1), perdidas, marker='o')
            plt.title("Pérdida por Época")
            plt.xlabel("Época")
            plt.ylabel("Pérdida acumulada")
            plt.grid(True)
            mostrar_y_guardar_grafico(fig, "Pérdida por Época")
    elif opcion == '4':
        if not etiquetas_test:
            print("Evalúa el modelo primero (opción 2).")
        else:
            matriz = confusion_matrix(etiquetas_test, predicciones_test)
            fig = plt.figure(figsize=(6, 5))
            sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', xticklabels=dataset_completo.classes, yticklabels=dataset_completo.classes)
            plt.xlabel("Predicción")
            plt.ylabel("Etiqueta Real")
            plt.title("Matriz de Confusión")
            mostrar_y_guardar_grafico(fig, "Matriz de Confusión")
    elif opcion == '5':
        dataiter = iter(cargador_prueba)
        imagenes, etiquetas = next(dataiter)
        imagenes = imagenes.to(dispositivo)
        salida = modelo(imagenes)
        _, pred = torch.max(salida, 1)
        imagenes = imagenes.cpu() * 0.5 + 0.5
        fig = plt.figure(figsize=(12, 6))
        for idx in range(6):
            ax = fig.add_subplot(2, 3, idx+1, xticks=[], yticks=[])
            img = np.transpose(imagenes[idx].numpy(), (1, 2, 0))
            ax.imshow(img)
            ax.set_title(f"Real: {etiquetas[idx]} / Pred: {pred[idx].item()}")
        plt.tight_layout()
        mostrar_y_guardar_grafico(fig, "Muestras Clasificadas")
    elif opcion == '6':
        validacion_cruzada(dataset_completo, k=5)
    elif opcion == '7':
        print("Saliendo del programa. ¡Hasta pronto!")
        break
    else:
        print("Opción no válida. Intenta con un número del 1 al 7.")



MENÚ DEL PROYECTO DE ANÁLISIS DE RAYOS X
1. Entrenar el modelo
2. Evaluar el modelo
3. Ver gráfico de pérdida
4. Ver matriz de confusión
5. Mostrar imágenes clasificadas
6. Validación cruzada
7. Salir



KeyboardInterrupt: Interrupted by user